In [2]:
# Cloning GitHub repo
!git clone https://github.com/Romit-M/UIDAI-Hackathon-2026.git
%cd UIDAI-Hackathon-2026

# IMPORTS
!pip install rapidfuzz -q
from rapidfuzz import process, utils

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# HELPER FUNCTIONS

# Data loader
def load_data(file_path):
  df = pd.read_csv(file_path)
  df.columns = df.columns.str.lower()
  return df

# Getting file names
def get_filename(category, r):
  return f"api_data_aadhar_{category}_{r}.csv"


Cloning into 'UIDAI-Hackathon-2026'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 93 (delta 16), reused 33 (delta 11), pack-reused 51 (from 1)
Receiving objects: 100% (93/93), 100.06 MiB | 12.48 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Updating files: 100% (25/25), done.
/content/UIDAI-Hackathon-2026
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.5 MB/s eta 0:00:00


### **DATA CLEANING**

In [3]:
# TEXT NORMALIZATION (making text comparable)

import re

def normalize_text(s):
    if not isinstance(s, str):
        return s

    s = s.lower().strip()
    s = s.replace('&', 'and').replace(' ', '')

    # Remove punctuation
    s = re.sub(r'[^a-z0-9]', '', s)

    return s

In [4]:
# UNIQUE STATE MAPPING USING FUZZY LOGIC

# Manual override for historic name changes
state_manual_fixes = {
    'orissa': 'odisha',
    'pondicherry': 'puducherry',
    'uttaranchal': 'uttarakhand',
    'damananddiu': 'dadraandnagarhavelianddamananddiu',
    'dadraandnagarhaveli': 'dadraandnagarhavelianddamananddiu'
}

# Official Aadhaar/UIDAI standard list
official_states = [
    'haryana', 'bihar', 'jammuandkashmir', 'tamilnadu', 'maharashtra',
    'gujarat', 'odisha', 'westbengal', 'kerala', 'rajasthan', 'punjab',
    'himachalpradesh', 'uttarpradesh', 'assam', 'uttarakhand',
    'madhyapradesh', 'karnataka', 'andhrapradesh', 'telangana', 'goa',
    'nagaland', 'jharkhand', 'delhi', 'chhattisgarh', 'meghalaya',
    'chandigarh', 'puducherry', 'manipur', 'sikkim', 'tripura',
    'mizoram', 'arunachalpradesh', 'ladakh', 'lakshadweep',
    'dadraandnagarhavelianddamananddiu', 'andamanandnicobarislands'
]

def state_mapping(df):

  # Manual override
  df['state'] = df['state'].replace(state_manual_fixes)

  # Find closest match (Fuzzy matching)
  def get_closest_match(state):
      # Returns the best match from official_states if score > 80%
      match = process.extractOne(state, official_states, score_cutoff=80)
      return match[0] if match else state

  # Get unique dirty states
  dirty_states = df['state'].unique()

  # Create and apply an automated mapping dictionary
  auto_mapping = {ds: get_closest_match(ds) for ds in dirty_states}
  df['state'] = df['state'].map(auto_mapping)

  return df


In [11]:
# UNIQUE DISTRICT MAPPING USING FUZZY LOGIC

district_manual_fixes = {
	("maharashtra", "osmanabad"): "dharashiv",
	("uttarpradesh", "allahabad"): "prayagraj",
	("uttarpradesh", "faizabad"): "ayodhya",
  ("westbengal", "north24parganas"): "northtwentyfourparganas",
  ("westbengal", "south24parganas"): "southtwentyfourparganas",
	}

official_districts = {
    "andamanandnicobarislands": [
        "nicobar", "northandmiddleandaman", "southandaman"
    ],
    "andhrapradesh": [
        "allurisitharamaraju", "anakapalli", "anantapuramu", "annamayya", "bapatla",
        "chittoor", "dr.b.r.ambedkarkonaseema", "eastgodavari", "eluru", "guntur",
        "kakinada", "krishna", "kurnool", "nandyal", "ntr", "palnadu", "parvathipurammanyam",
        "prakasam", "sripottisriramulunellore", "srisatyasai", "srikakulam",
        "tirupati", "visakhapatnam", "vizianagaram", "westgodavari", "ysrkadapa"
    ],
    "arunachalpradesh": [
        "anjaw", "changlang", "dibangvalley", "eastkameng", "eastsiang", "itanagar",
        "kamle", "kradaadi", "kurungkumey", "leparada", "lohit", "longding",
        "lowerdibangvalley", "lowersiang", "lowersubansiri", "namsai", "pakkekessang",
        "papumpare", "shiyomi", "siang", "tawang", "tirap", "uppersiang",
        "uppersubansiri", "westkameng", "westsiang"
    ],
    "assam": [
        "bajali", "baksa", "barpeta", "biswanath", "bongaigaon", "cachar", "charaideo",
        "chirang", "darrang", "dhemaji", "dhubri", "dibrugarh", "dimahasao", "goalpara",
        "golaghat", "hailakandi", "hojai", "jorhat", "kamrup", "kamrupmetropolitan",
        "karbianglong", "karimganj", "kokrajhar", "lakhimpur", "majuli", "morigaon",
        "nagaon", "nalbari", "sivasagar", "sonitpur", "southsalmaramankachar",
        "tamulpur", "tinsukia", "udalguri", "westkarbianglong"
    ],
    "bihar": [
        "araria", "arwal", "aurangabad", "banka", "begusarai", "bhagalpur", "bhojpur",
        "buxar", "darbhanga", "eastchamparan", "gaya", "gopalganj", "jamui", "jehanabad",
        "kaimur", "katihar", "khagaria", "kishanganj", "lakhisarai", "madhepura",
        "madhubani", "munger", "muzaffarpur", "nalanda", "nawada", "patna", "purnea",
        "rohtas", "saharsa", "samastipur", "saran", "sheikhpura", "sheohar", "sitamarhi",
        "siwan", "supaul", "vaishali", "westchamparan"
    ],
    "chandigarh": [
        "chandigarh"
    ],
    "chhattisgarh": [
        "balod", "balodabazar", "balrampur", "bastar", "bemetara", "bijapur", "bilaspur",
        "dantewada", "dhamtari", "durg", "gariaband", "gaurelapendramarwahi",
        "janjgirchampa", "jashpur", "kabirdham", "kanker", "khairagarhchhuikhadangandai",
        "kondagaon", "korba", "korea", "mahasamund", "manendragarhchirmiribharatpur",
        "mohlamanpurambagarhchowki", "mungeli", "narayanpur", "raigarh", "raipur",
        "rajnandgaon", "sarangarhbilaigarh", "shakti", "sukma", "surajpur", "surguja"
    ],
    "dadraandnagarhavelianddamananddiu": [
        "dadraandnagarhaveli", "daman", "diu"
    ],
    "delhi": [
        "centraldelhi", "eastdelhi", "newdelhi", "northdelhi", "northeastdelhi",
        "northwestdelhi", "shahdara", "southdelhi", "southeastdelhi", "southwestdelhi",
        "westdelhi"
    ],
    "goa": [
        "northgoa", "southgoa"
    ],
    "gujarat": [
        "ahmedabad", "amreli", "anand", "aravalli", "banaskantha", "bharuch", "bhavnagar",
        "botad", "chhotaudepur", "dahod", "devbhumidwarka", "gandhinagar", "girsomnath",
        "jamnagar", "junagadh", "kheda", "kutch", "mahesana", "mahisagar", "morbi",
        "narmada", "navsari", "panchmahal", "patan", "porbandar", "rajkot", "sabarkantha",
        "surat", "surendranagar", "tapi", "thedangs", "vadodara", "valsad"
    ],
    "haryana": [
        "ambala", "bhiwani", "charkhidadri", "faridabad", "fatehabad", "gurugram",
        "hisar", "jhajjar", "jind", "kaithal", "karnal", "kurukshetra", "mahendragarh",
        "nuh", "palwal", "panchkula", "panipat", "rewari", "rohtak", "sirsa", "sonipat",
        "yamunanagar"
    ],
    "himachalpradesh": [
        "bilaspur", "chamba", "hamirpur", "kangra", "kinnaur", "kullu", "lahaulandspiti",
        "mandi", "shimla", "sirmaur", "solan", "una"
    ],
    "jammuandkashmir": [
        "anantnag", "bandipora", "baramulla", "budgam", "doda", "ganderbal", "jammu",
        "kathua", "kishtwar", "kulgam", "kupwara", "poonch", "pulwama", "rajouri",
        "ramban", "reasi", "samba", "shopian", "srinagar", "udhampur"
    ],
    "jharkhand": [
        "bokaro", "chatra", "deoghar", "dhanbad", "dumka", "eastsinghbhum", "garhwa",
        "giridih", "godda", "gumla", "hazaribagh", "jamtara", "khunti", "koderma",
        "latehar", "lohardaga", "pakur", "palamu", "ramgarh", "ranchi", "sahibganj",
        "seraikelakharsawan", "simdega", "westsinghbhum"
    ],
    "karnataka": [
        "bagalkot", "ballari", "belagavi", "bengalururural", "bengaluruurban", "bidar",
        "chamarajanagar", "chikkaballapura", "chikkamagaluru", "chitradurga",
        "dakshinakannada", "davanagere", "dharwad", "gadag", "hassan", "haveri",
        "kalaburagi", "kodagu", "kolar", "koppal", "mandya", "mysuru", "raichur",
        "ramanagara", "shivamogga", "tumakuru", "udupi", "uttarakannada", "vijayanagar",
        "vijayapura", "yadgir"
    ],
    "kerala": [
        "alappuzha", "ernakulam", "idukki", "kannur", "kasaragod", "kollam", "kottayam",
        "kozhikode", "malappuram", "palakkad", "pathanamthitta", "thiruvananthapuram",
        "thrissur", "wayanad"
    ],
    "ladakh": [
        "drass", "kargil", "leh", "sham"
    ],
    "lakshadweep": [
        "lakshadweep"
    ],
    "madhyapradesh": [
        "agarmalwa", "alirajpur", "anuppur", "ashoknagar", "balaghat", "barwani", "betul",
        "bhind", "bhopal", "burhanpur", "chhatarpur", "chhindwara", "damoh", "datia",
        "dewas", "dhar", "dindori", "guna", "gwalior", "harda", "hoshangabad", "indore",
        "jabalpur", "jhabua", "katni", "khandwa", "khargone", "mandla", "mandsaur",
        "mauganj", "morena", "narmadapuram", "narsinghpur", "neemuch", "niwari", "panna",
        "raisen", "rajgarh", "ratlam", "rewa", "sagar", "satna", "sehore", "seoni",
        "shahdol", "shajapur", "sheopur", "shivpuri", "sidhi", "singrauli", "tikamgarh",
        "ujjain", "umaria", "vidisha"
    ],
    "maharashtra": [
        "ahmednagar", "akola", "amravati", "aurangabad", "beed", "bhandara", "buldhana",
        "chandrapur", "dhule", "gadchiroli", "gondia", "hingoli", "jalgaon", "jalna",
        "kolhapur", "latur", "mumbaicity", "mumbaisuburban", "nagpur", "nanded",
        "nandurbar", "nashik", "osmanabad", "palghar", "parbhani", "pune", "raigad",
        "ratnagiri", "sangli", "satara", "sindhudurg", "solapur", "thane", "wardha",
        "washim", "yavatmal"
    ],
    "manipur": [
        "bishnupur", "chandel", "churachandpur", "imphaleast", "imphalwest", "jiribam",
        "kakching", "kamjong", "kangpokpi", "noney", "pherzawl", "senapati", "tamenglong",
        "tengnoupal", "thoubal", "ukhrul"
    ],
    "meghalaya": [
        "eastgarohills", "eastjaintiahills", "eastkhasihills", "mairang", "northgarohills",
        "ribhoi", "southgarohills", "southwestgarohills", "southwestkhasihills",
        "westgarohills", "westjaintiahills", "westkhasihills"
    ],
    "mizoram": [
        "aizawl", "champhai", "hnahthial", "khawzawl", "kolasib", "langlei", "lawngtlai",
        "mamit", "saiha", "saitual", "serchhip"
    ],
    "nagaland": [
        "chumukedima", "dimapur", "kiphire", "kohima", "longleng", "mokokchung", "mon",
        "niuland", "noklak", "peren", "phek", "shamator", "tseminyu", "tuensang",
        "wokha", "zunheboto"
    ],
    "odisha": [
        "angul", "balangir", "balasore", "bargarh", "bhadrak", "boudh", "cuttack",
        "deogarh", "dhenkanal", "gajapati", "ganjam", "jagatsinghpur", "jajpur",
        "jharsuguda", "kalahandi", "kandhamal", "kendrapara", "kendujhar", "khordha",
        "koraput", "malkangiri", "mayurbhanj", "nabarangpur", "nayagarh", "nuapada",
        "puri", "rayagada", "sambalpur", "subarnapur", "sundargarh"
    ],
    "puducherry": [
        "karaikal", "mahe", "puducherry", "yanam"
    ],
    "punjab": [
        "amritsar", "barnala", "bathinda", "faridkot", "fatehgarhsahib", "fazilka",
        "ferozepur", "gurdaspur", "hoshiarpur", "jalandhar", "kapurthala", "ludhiana",
        "malerkotla", "mansa", "moga", "pathankot", "patiala", "rupnagar",
        "sahibzadaajitsinghnagar", "sangrur", "shahidbhagatsinghnagar",
        "srimuktsarsahib", "tarntaran"
    ],
    "rajasthan": [
        "ajmer", "alwar", "anupgarh", "balotra", "banswara", "baran", "barmer", "beawar",
        "bharatpur", "bhilwara", "bikaner", "bundi", "chittorgarh", "churu", "dausa",
        "dholpur", "didwanakuchaman", "dudu", "dungarpur", "gangapurcity", "hanumangarh",
        "jaipur", "jaipurrural", "jaisalmer", "jalore", "jhalawar", "jhunjhunu",
        "jodhpur", "jodhpurrural", "karauli", "kekri", "khairthaltijara", "kota",
        "kotputlibehror", "nagaur", "neemkathana", "pali", "phalodi", "pratapgarh",
        "rajsamand", "salumbar", "sanchore", "sawaimadhopur", "shahpura", "sikar",
        "sirohi", "sriganganagar", "tonk", "udaipur"
    ],
    "sikkim": [
        "gangtok", "gyalshing", "mangan", "namchi", "pakyong", "soreng"
    ],
    "tamilnadu": [
        "ariyalur", "chengalpattu", "chennai", "coimbatore", "cuddalore", "dharmapuri",
        "dindigul", "erode", "kallakurichi", "kancheepuram", "kanniyakumari", "karur",
        "krishnagiri", "madurai", "mayiladuthurai", "nagapattinam", "namakkal",
        "nilgiris", "perambalur", "pudukkottai", "ramanathapuram", "ranipet", "salem",
        "sivaganga", "tenkasi", "thanjavur", "theni", "thoothukudi", "tiruchirappalli",
        "tirunelveli", "tirupathur", "tiruvannamalai", "tiruvarur", "vellore",
        "viluppuram", "virudhunagar"
    ],
    "telangana": [
        "adilabad", "bhadradrikothagudem", "hanumakonda", "hyderabad", "jagtial",
        "jangaon", "jayashankarbhupalpally", "jogulambagadwal", "kamareddy",
        "karimnagar", "khammam", "komarambheem", "mahabubabad", "mahabubnagar",
        "mancherial", "medak", "medchalmalkajgiri", "mulugu", "nagarkurnool",
        "nalgonda", "narayanpet", "nirmal", "nizamabad", "peddapalli", "rajannasircilla",
        "rangareddy", "sangareddy", "siddipet", "suryapet", "vikarabad", "wanaparthy",
        "warangal", "yadadribhuvanagiri"
    ],
    "tripura": [
        "dhalai", "gomati", "khowai", "northtripura", "sepahijala", "southtripura",
        "unakoti", "westtripura"
    ],
    "uttarpradesh": [
        "agra", "aligarh", "allahabadprayagraj", "ambedkarnagar", "amethi", "amroha",
        "auraiya", "ayodhya", "azamgarh", "badaun", "bahraich", "ballia", "balrampur",
        "banda", "barabanki", "bareilly", "basti", "bhadohi", "bijnor", "budaun",
        "bulandshahr", "chandauli", "chitrakoot", "deoria", "etah", "etawah",
        "farrukhabad", "fatehpur", "firozabad", "gautambuddhanagar", "ghaziabad",
        "ghazipur", "gonda", "gorakhpur", "hamirpur", "hapur", "hardoi", "hathras",
        "jalaun", "jaunpur", "jhansi", "kannauj", "kanpurdehat", "kanpurnagar",
        "kasganj", "kaushambi", "kheri", "kushinagar", "lalitpur", "lucknow",
        "maharajganj", "mahoba", "mainpuri", "mathura", "mau", "meerut", "mirzapur",
        "moradabad", "muzaffarnagar", "pilibhit", "pratapgarh", "raebareli", "rampur",
        "saharanpur", "sambhal", "santkabirnagar", "shahjahanpur", "shamli",
        "shravasti", "siddharthnagar", "sitapur", "sonbhadra", "sultanpur", "unnao",
        "varanasi"
    ],
    "uttarakhand": [
        "almora", "bageshwar", "chamoli", "champawat", "dehradun", "haridwar", "nainital",
        "paurigarhwal", "pithoragarh", "rudraprayag", "tehrigarhwal", "udhamsinghnagar",
        "uttarkashi"
    ],
    "westbengal": [
        "alipurduar", "bankura", "birbhum", "coochbehar", "dakshindinajpur", "darjeeling",
        "hooghly", "howrah", "jalpaiguri", "jhargram", "kalimpong", "kolkata", "malda",
        "murshidabad", "nadia", "north24parganas", "paschimbardhaman", "paschimmedinipur",
        "purbabardhaman", "purbamedinipur", "purulia", "south24parganas", "uttardinajpur"
    ]
}

def district_mapping(df):

    # Get unique combinations of 'State + District' pairs
    unique_combinations = df[['state', 'district']].drop_duplicates()

    # Mapping key
    district_map = {}

    # iterate through each unique combination
    for _, row in unique_combinations.iterrows():
        st, dt = row['state'], row['district']
        current_pair = (st, dt)

        # Manual overrides
        if current_pair in district_manual_fixes:
            district_map[current_pair] = district_manual_fixes[current_pair]
            continue

        # Find closest match (Fuzzy matching)
        if st in official_districts:
            choices = official_districts[st]
            # Match dirty district against specific state-official-districts
            match = process.extractOne(dt, choices, score_cutoff=85)

            if match:
                district_map[current_pair] = match[0]
            else:
                district_map[current_pair] = dt # Flag for review if needed
        else:
            district_map[current_pair] = dt

    # Applying District Mapping ket to the data
    df['district'] = df.apply(lambda x: district_map.get((x['state'], x['district']), x['district']), axis=1)
    return df


In [12]:
# CLEANING PIPELINE

def clean_pipeline(df):
  df['state'] = df['state'].astype(str).apply(normalize_text)
  df['district'] = df['district'].astype(str).apply(normalize_text)

  df = state_mapping(df)
  df = district_mapping(df)

  return df


In [13]:
# DATA LOADING -> CLEANING -> SAVING CLEANED DATA

categories = ['biometric', 'demographic', 'enrolment']
ranges = ['0_500000', '500000_1000000']

RAW_PATH = "data/raw/api_data_aadhar"
PROCESSED_PATH = "data/processed/api_data_aadhar"

for category in categories:
  for r in ranges:

    # Load file
    filename = get_filename(category, r)
    file_path = f"{RAW_PATH}_{category}/{filename}"
    df = load_data(file_path)

    # Apply cleaning
    df = clean_pipeline(df)

    # Save cleaned file
    output_filename = filename.replace('.csv', '_cleaned.csv')
    output_path = f"{PROCESSED_PATH}_{category}/{output_filename}"
    df.to_csv(output_path, index=False)

    print(f"{output_filename} saved.")


api_data_aadhar_biometric_0_500000_cleaned.csv saved.
api_data_aadhar_biometric_500000_1000000_cleaned.csv saved.
api_data_aadhar_demographic_0_500000_cleaned.csv saved.
api_data_aadhar_demographic_500000_1000000_cleaned.csv saved.
api_data_aadhar_enrolment_0_500000_cleaned.csv saved.
api_data_aadhar_enrolment_500000_1000000_cleaned.csv saved.


In [14]:
# PUSH CLEANED FILES TO REPO

from google.colab import userdata
pat = userdata.get('GitHubAccessToken')

!git config --global user.name "Romit-M"
!git config --global user.email "romitrmaity@gmail.com"

!git add .
!git commit -m "Added cleaned data files"
!git push https://{pat}@github.com/Romit-M/UIDAI-Hackathon-2026.git


[main 19a5632] Added cleaned data files
 6 files changed, 293946 insertions(+), 293946 deletions(-)
Enumerating objects: 20, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 19.15 MiB | 1.30 MiB/s, done.
Total 13 (delta 9), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (9/9), completed with 4 local objects.
To https://github.com/Romit-M/UIDAI-Hackathon-2026.git
   a144d08..19a5632  main -> main


In [1]:
# !rm -rf /content/UIDAI-Hackathon-2026
# %cd /content


/content
